<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/DDQN3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

project_files = [
    "data_preprocessing.py",
    "environment.py",
    "scenario_generator.py",
    "reward.py",
    "agent.py",
    "train.py",
    "schedule_generator.py",
    "evaluation.py",
    "utils.py",
    "config.py"
]

base_path = "/content/rl_home_care_scheduler"

os.makedirs(base_path, exist_ok=True)

for file in project_files:
    path = os.path.join(base_path, file)
    if not os.path.exists(path):
        with open(path, "w") as f:
            f.write(f"# {file}\n")
            f.write("# Auto-generated module placeholder\n\n")

print("Project structure created at:", base_path)
print("\nFiles:")
for f in project_files:
    print("-", f)

Project structure created at: /content/rl_home_care_scheduler

Files:
- data_preprocessing.py
- environment.py
- scenario_generator.py
- reward.py
- agent.py
- train.py
- schedule_generator.py
- evaluation.py
- utils.py
- config.py


In [3]:
import sys
import os

# Add the project's base directory to sys.path
base_path = "/content/rl_home_care_scheduler"
sys.path.insert(0, base_path)

from data_preprocessing import preprocess_pipeline

DATA_PATH = "/content/rl_home_care_scheduler/data"

data = preprocess_pipeline(
    carers_path=f"{DATA_PATH}/carers_available.csv",
    visits_path=f"{DATA_PATH}/visits.csv"
)

carers = data["carers"]
visits = data["visits"]
distance_matrix = data["distance_matrix"]

print(carers.shape, visits.shape)

(462, 28) (3118, 21)


In [4]:
import pandas as pd

carers = pd.read_csv("/content/rl_home_care_scheduler/data/carers_available.csv")
visits = pd.read_csv("/content/rl_home_care_scheduler/data/visits.csv")

print("Carers:", carers.shape)
print("Visits:", visits.shape)
print(carers.columns[:5])
print(visits.columns[:5])

Carers: (462, 23)
Visits: (3118, 15)
Index(['CarerID', 'WorkerType', 'Latitude', 'Longitude',
       'PreferredAreaLatitude'],
      dtype='object')
Index(['VisitID', 'PatientID', 'Day', 'Latitude', 'Longitude'], dtype='object')


In [6]:
from data_preprocessing import preprocess_pipeline
from environment import HomeCareEnv
from state_encoder import StateEncoder
from agent import DoubleDQNAgent

# 1. Load data again
data = preprocess_pipeline(
    "/content/rl_home_care_scheduler/data/carers_available.csv",
    "/content/rl_home_care_scheduler/data/visits.csv"
)

carers = data["carers"]
visits = data["visits"]
distance_matrix = data["distance_matrix"]

# 2. Recreate environment
environment = HomeCareEnv(
    carers,
    visits,
    distance_matrix
)

# 3. Recreate encoder
encoder = StateEncoder(
    environment.n_carers,
    environment.n_visits
)

state_dim = encoder.state_size()

# 4. Recreate agent (IMPORTANT: must match trained model)
agent = DoubleDQNAgent(
    state_dim=state_dim,
    n_visits=environment.n_visits
)

# 5. Load trained model
agent.load("/content/rl_home_care_scheduler/best_ddqn_model.pth")

print("Environment, Agent, Encoder recreated successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/rl_home_care_scheduler/best_ddqn_model.pth'

In [7]:
import os

project_dir = "/content/rl_home_care_scheduler"
files = os.listdir(project_dir)
for f in files:
    print(f)

__pycache__
state_encoder.py
scenario_generator.py
schedule_generator.py
data
evaluation.py
config.py
.ipynb_checkpoints
environment.py
agent.py
rolling_horizon.py
data_preprocessing.py
utils.py
reward.py
train.py


In [11]:
# schedule_generator.py

import numpy as np
import pandas as pd

from state_encoder import StateEncoder


class ScheduleGenerator:

    def __init__(
            self,
            env,
            agent,
            encoder):

        self.env = env
        self.agent = agent
        self.encoder = encoder

    # =====================================================
    # GENERATE SCHEDULE (SAFE VERSION)
    # =====================================================
    def generate(self):

        state = self.env.reset()

        schedule = []
        done = False

        # IMPORTANT: prevents infinite loops
        max_steps = self.env.n_visits * 2
        step = 0

        # IMPORTANT: inference mode
        self.agent.epsilon = 0.0

        while not done and step < max_steps:

            # ---------------------------------
            # VISIT MASK
            # ---------------------------------
            visit_mask = self.env.get_visit_mask()

            if np.sum(visit_mask) == 0:
                break

            # ---------------------------------
            # ENCODE STATE
            # ---------------------------------
            encoded_state = self.encoder.encode(state)

            # ---------------------------------
            # SELECT VISIT (DDQN)
            # ---------------------------------
            visit = self.agent.select_action(
                encoded_state,
                visit_mask
            )

            if visit is None:
                break

            # ---------------------------------
            # ENV STEP
            # (carer selected internally)
            # ---------------------------------
            next_state, reward, done, info = self.env.step(visit)

            state = next_state

            step += 1

        # =====================================================
        # BUILD OUTPUT DATAFRAME
        # =====================================================
        schedule_df = pd.DataFrame(self.env.schedule)

        if len(schedule_df) > 0:
            assigned_visits = set(schedule_df["VisitID"])
        else:
            assigned_visits = set()

        all_visits = set(self.env.visits["VisitID"])

        unassigned_ids = list(all_visits - assigned_visits)

        unassigned_df = self.env.visits[
            self.env.visits["VisitID"].isin(unassigned_ids)
        ]

        return schedule_df, unassigned_df

    # =====================================================
    # SAVE RESULTS
    # =====================================================
    def save_results(
            self,
            schedule_df,
            unassigned_df,
            path):

        schedule_df.to_csv(
            f"{path}/assigned_schedule.csv",
            index=False
        )

        unassigned_df.to_csv(
            f"{path}/unassigned_visits.csv",
            index=False
        )

        print("\n✅ Schedule saved successfully")
        print(f"Assigned visits: {len(schedule_df)}")
        print(f"Unassigned visits: {len(unassigned_df)}")

In [12]:
visit_mask = environment.get_visit_mask()

print("Total visits:", len(visit_mask))
print("Valid visits:", np.sum(visit_mask))
print("Invalid visits:", len(visit_mask) - np.sum(visit_mask))

Total visits: 3118
Valid visits: 2929.0
Invalid visits: 189.0


In [13]:
print("Sample feasibility check:")

for i in range(min(5, environment.n_visits)):
    feas = environment.get_feasible_carers(i)
    print(f"Visit {i} feasible carers:", len(feas))

Sample feasibility check:
Visit 0 feasible carers: 100
Visit 1 feasible carers: 84
Visit 2 feasible carers: 84
Visit 3 feasible carers: 84
Visit 4 feasible carers: 84


In [14]:
valid_visits = np.where(visit_mask == 1)[0]

# fallback heuristic: pick first valid visit
visit = valid_visits[0] if len(valid_visits) > 0 else None

In [21]:
state = environment.reset()

print("Before step:")
print("Scheduled:", len(environment.schedule))

# pick first visit
visit_mask = environment.get_visit_mask()
valid_visits = np.where(visit_mask == 1)[0]

print("Valid visits:", len(valid_visits))

if len(valid_visits) > 0:
    v = valid_visits[0]

    next_state, reward, done, info = environment.step(v)

    print("\nAfter step:")
    print("Reward:", reward)
    print("Scheduled:", len(environment.schedule))
    print("First entry:", environment.schedule[0] if len(environment.schedule) > 0 else None)
else:
    print("No valid visits found")

Before step:
Scheduled: 0
Valid visits: 3118

After step:
Reward: 8.77737756648276
Scheduled: 1
First entry: {'CarerID': 'C0094', 'VisitID': 'V000001', 'PatientID': 'P0001', 'Distance': np.float64(1.2226224335172389), 'Workload': np.float32(0.75)}


In [16]:
print(environment.carers.columns)
print(environment.visits.columns)

Index(['CarerID', 'WorkerType', 'Latitude', 'Longitude',
       'PreferredAreaLatitude', 'PreferredAreaLongitude',
       'MaxTravelDistanceMiles', 'ExperienceInYears', 'Skills', 'Gender',
       'Languages', 'VehicleType', 'WorkingDays', 'OffDays', 'ShiftStart',
       'ShiftEnd', 'ContractHoursPerWeek', 'MaxDailyHours', 'MaxWeeklyHours',
       'PreferredShift', 'Date', 'DayOfWeek', 'Status', 'ShiftStartMin',
       'ShiftEndMin', 'LatNorm', 'LonNorm', 'SkillCount'],
      dtype='object')
Index(['VisitID', 'PatientID', 'Day', 'Latitude', 'Longitude',
       'VisitDurationMinutes', 'TimeWindowStart', 'TimeWindowEnd',
       'PriorityLevel', 'ComplexityLevel', 'RequiredSkills', 'PreferredGender',
       'PreferredLanguage', 'TwoCarerRequired', 'ContinuityRequirement',
       'TimeWindowStartMin', 'TimeWindowEndMin', 'LatNorm', 'LonNorm',
       'RequiredSkillCount', 'VisitDurationHours'],
      dtype='object')


In [ ]:
from schedule_generator import ScheduleGenerator
from evaluation import SchedulerEvaluator

# Instantiate and run the schedule generator to define 'schedule' and 'unassigned'
generator = ScheduleGenerator(
    environment,
    agent,
    encoder
)

schedule, unassigned = generator.generate()

generator.save_results(
    schedule,
    unassigned,
    "/content/rl_home_care_scheduler"
)

print("\n===== FINAL RESULTS ====")
print("Assigned visits:", len(schedule))
print("Unassigned visits:", len(unassigned))

if len(schedule) > 0:
    print("\nSample assignments:")
    print(schedule.head(3))

# Original evaluation code
evaluator = SchedulerEvaluator(environment)

metrics = evaluator.evaluate(schedule, unassigned)

print(metrics)

evaluator.save_results(
    metrics,
    "/content/rl_home_care_scheduler"
)